# 📘 Chapter 01a — Data Pipeline: Text → Tensors → GPU
**Series: Understanding Transformers from Scratch**

---

## What this chapter covers
All the **plumbing** a language model needs before any learning happens:
- Text → characters → integers (tokenization)
- Integers → PyTorch tensors
- What `dtype` means and why `torch.long` matters
- How to move tensors to GPU with `.to(device)`
- How training batches are built (`x` inputs, `y` targets)

**No model yet — just data handling.**
In Chapter 01b we'll use everything built here to train a real model.

---

## The Full Data Flow

```
  "hello world"          ← raw text (a Python string)
       │
       ▼  1. Find unique characters
  {' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w'}   ← vocabulary (a Python set)
       │
       ▼  2. Sort for stability
  [' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w']   ← sorted list
       │
       ▼  3. Build lookup dicts
  stoi = {'h':3, 'e':2, 'l':4, ...}           ← string→int
  itos = {3:'h', 2:'e', 4:'l', ...}           ← int→string
       │
       ▼  4. Encode full text
  [3, 2, 4, 4, 5, 0, 7, 5, 6, 4, 1]          ← Python list of ints
       │
       ▼  5. Convert to tensor
  tensor([3, 2, 4, 4, 5, 0, 7, 5, 6, 4, 1])  ← PyTorch 1D tensor (CPU)
       │
       ▼  6. Move to device (GPU if available)
  tensor([...]) on cuda:0                      ← same data, now in GPU memory
       │
       ▼  7. Build batches
  x: (batch, block_size) ← inputs
  y: (batch, block_size) ← targets (x shifted +1)
```

---
## Section 1 — What is a Python `set`?

In [1]:
import sys
print(sys.executable)

e:\Installation\llm\Scripts\python.exe


In [2]:
from collections import Counter
import string
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim

In [3]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "tinyshakespeare.txt")

('tinyshakespeare.txt', <http.client.HTTPMessage at 0x189a9d08b60>)

In [4]:
# read the file: 


with open("tinyshakespeare.txt", "r", encoding="utf-8") as dataset:
    txt = dataset.read().replace('\n', '')
    char_counts = Counter(txt)
    # words = []
    # # get the lines.
    # for line in dataset:
    #     # break the lines into words
    chars = set(txt) | set(string.ascii_letters) |set(string.digits) | set(string.punctuation)    # ensure all chars are there

chars = sorted(chars)
print(chars)

#char_counts = Counter(chars)
#print(char_counts)

all_chars = string.ascii_letters + string.digits + string.punctuation

for ch in all_chars:
    char_counts.setdefault(ch,0)

for ch, count in char_counts.items():
    print(f"{repr(ch)}: {count}")


print(len(chars))


## we have the character set 
## we have all the chars and their counts, and we have not left english alphabet too.
    

[' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}', '~']
'F': 1797
'i': 45537
'r': 48889
's': 49696
't': 67009
' ': 169892
'C': 3820
'z': 356
'e': 94611
'n': 48529
':': 10316
'B': 2761
'f': 15770
'o': 65798
'w': 17585
'p': 10808
'c': 15623
'd': 31358
'a': 55507
'y': 20448
'u': 26584
'h': 51310
',': 19846
'm': 22243
'k': 7088
'.': 7885
'A': 7819
'l': 33339
'S': 4523
'Y': 1718
'v': 7793
'?': 2462
'R': 4869
'M': 2840
'W': 3530
"'": 6187
'L': 3876
'I': 11832
'N': 5079
'g': 13356
';': 3628
'b': 11321
'!': 2172
'O': 5481
'j': 628
'V': 798
'-': 1897
'T': 7015
'H': 3068
'E': 6041
'

---
## Section 2 — get the index and character map.

In [5]:
# sorted(iterable)
#   Takes ANY iterable (list, set, string, ...) and returns a SORTED LIST.
#   For strings/characters: sorts by ASCII value (space < digits < uppercase < lowercase)
#   WHY sort? Sets are unordered — each Python run may produce a different order.
#   Sorting gives a DETERMINISTIC vocabulary: same text → same mapping every run.


print()
print("Character order (by ASCII value):")
for i, ch in enumerate(chars):
    print(f"  index {i}: '{ch}'  (ASCII {ord(ch)})")


Character order (by ASCII value):
  index 0: ' '  (ASCII 32)
  index 1: '!'  (ASCII 33)
  index 2: '"'  (ASCII 34)
  index 3: '#'  (ASCII 35)
  index 4: '$'  (ASCII 36)
  index 5: '%'  (ASCII 37)
  index 6: '&'  (ASCII 38)
  index 7: '''  (ASCII 39)
  index 8: '('  (ASCII 40)
  index 9: ')'  (ASCII 41)
  index 10: '*'  (ASCII 42)
  index 11: '+'  (ASCII 43)
  index 12: ','  (ASCII 44)
  index 13: '-'  (ASCII 45)
  index 14: '.'  (ASCII 46)
  index 15: '/'  (ASCII 47)
  index 16: '0'  (ASCII 48)
  index 17: '1'  (ASCII 49)
  index 18: '2'  (ASCII 50)
  index 19: '3'  (ASCII 51)
  index 20: '4'  (ASCII 52)
  index 21: '5'  (ASCII 53)
  index 22: '6'  (ASCII 54)
  index 23: '7'  (ASCII 55)
  index 24: '8'  (ASCII 56)
  index 25: '9'  (ASCII 57)
  index 26: ':'  (ASCII 58)
  index 27: ';'  (ASCII 59)
  index 28: '<'  (ASCII 60)
  index 29: '='  (ASCII 61)
  index 30: '>'  (ASCII 62)
  index 31: '?'  (ASCII 63)
  index 32: '@'  (ASCII 64)
  index 33: 'A'  (ASCII 65)
  index 34: 'B'  (ASCII

---
## Section 3 — Build the Vocabulary Mappings

In [6]:

vocab_size = len(chars)

# stoi = "string to integer"
# enumerate(chars) yields pairs: (0,' '), (1,'.'), (2,'a'), (3,'d'), ...
# {ch: i for i, ch in ...} builds a dict mapping each char to its index.
#
# Result: {' ':0, '.':1, 'a':2, 'd':3, 'e':4, 'f':5, 'h':6, 'l':7, ...}

stoi = {ch: i for i, ch in enumerate(chars)}

# itos = "integer to string" — the reverse
# Result: {0:' ', 1:'.', 2:'a', 3:'d', 4:'e', 5:'f', 6:'h', 7:'l', ...}

itos = {i: ch for i, ch in enumerate(chars)}

# encode: text → list of integers
encode = lambda s: [stoi[c] for c in s]

# decode: list of integers → text
decode = lambda l: ''.join([itos[i] for i in l])

print(f"Vocabulary ({vocab_size} tokens):")
for ch, idx in stoi.items():
    print(f"  '{ch}' → {idx}")

print(f"\nencode('hello') = {encode('hello')}")
print(f"decode({encode('hello')}) = '{decode(encode('hello'))}'")

Vocabulary (95 tokens):
  ' ' → 0
  '!' → 1
  '"' → 2
  '#' → 3
  '$' → 4
  '%' → 5
  '&' → 6
  ''' → 7
  '(' → 8
  ')' → 9
  '*' → 10
  '+' → 11
  ',' → 12
  '-' → 13
  '.' → 14
  '/' → 15
  '0' → 16
  '1' → 17
  '2' → 18
  '3' → 19
  '4' → 20
  '5' → 21
  '6' → 22
  '7' → 23
  '8' → 24
  '9' → 25
  ':' → 26
  ';' → 27
  '<' → 28
  '=' → 29
  '>' → 30
  '?' → 31
  '@' → 32
  'A' → 33
  'B' → 34
  'C' → 35
  'D' → 36
  'E' → 37
  'F' → 38
  'G' → 39
  'H' → 40
  'I' → 41
  'J' → 42
  'K' → 43
  'L' → 44
  'M' → 45
  'N' → 46
  'O' → 47
  'P' → 48
  'Q' → 49
  'R' → 50
  'S' → 51
  'T' → 52
  'U' → 53
  'V' → 54
  'W' → 55
  'X' → 56
  'Y' → 57
  'Z' → 58
  '[' → 59
  '\' → 60
  ']' → 61
  '^' → 62
  '_' → 63
  '`' → 64
  'a' → 65
  'b' → 66
  'c' → 67
  'd' → 68
  'e' → 69
  'f' → 70
  'g' → 71
  'h' → 72
  'i' → 73
  'j' → 74
  'k' → 75
  'l' → 76
  'm' → 77
  'n' → 78
  'o' → 79
  'p' → 80
  'q' → 81
  'r' → 82
  's' → 83
  't' → 84
  'u' → 85
  'v' → 86
  'w' → 87
  'x' → 88
  'y' →

## we will create a tensor with all the data and split them.

In [7]:
data = torch.tensor([stoi[c] for c in txt], dtype = torch.long)    ## creating a tensor from chars of input string.

# split the data for validation

n = int(0.9* len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Total tokens : {len(data)}")
print(f"Train tokens : {len(train_data)}")
print(f"Val tokens   : {len(val_data)}")

Total tokens : 1075394
Train tokens : 967854
Val tokens   : 107540


In [8]:
# let us set up the device now.

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Using device : {device}")

if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB") 

Using device : cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB


In [9]:
# for bigram, the block size is just 1 since we will use the previous token to predict the next token.

block_size = 8          # token the model sees at once.
batch_size = 32         # sequence por gradient step.



In [10]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))           # we generate random index values from where we will pick up the text. we need 32 of such values
    x = torch.stack([data[i  : i+block_size  ] for i in ix])            # we will get the values of x 
    y = torch.stack([data[i+1: i+block_size+1] for i in ix])            # This is our predictions.
    return x.to(device), y.to(device)

# test it
xb, yb = get_batch('train')
print(f"x shape : {xb.shape}   ← (batch={batch_size}, block={block_size})")
print(f"y shape : {yb.shape}")
print(f"device  : {xb.device}")

x shape : torch.Size([32, 8])   ← (batch=32, block=8)
y shape : torch.Size([32, 8])
device  : cuda:0


In [11]:
class BigramModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding_table = nn.Embedding(vocab_size, vocab_size)             #Registered as a pareameter in nn.Module

    def forward(self, idx, targets = None):
        # idx: (B, T) → logits: (B, T, vocab_size)
        logits = self.embedding_table(idx)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))

        return logits, loss


In [12]:
model = BigramModel(vocab_size).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

Parameters: 9025


In [13]:
## Training the model

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(20000):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 1000 == 0:
        print(f"step {step:5d} | loss: {loss.item():.4f}")

print(f"Final loss: {loss.item():.4f}")

step     0 | loss: 4.9827
step  1000 | loss: 4.0146
step  2000 | loss: 3.3127
step  3000 | loss: 2.8138
step  4000 | loss: 2.6993
step  5000 | loss: 2.6485
step  6000 | loss: 2.6707
step  7000 | loss: 2.5841
step  8000 | loss: 2.4158
step  9000 | loss: 2.5002
step 10000 | loss: 2.4948
step 11000 | loss: 2.6862
step 12000 | loss: 2.6031
step 13000 | loss: 2.4701
step 14000 | loss: 2.6079
step 15000 | loss: 2.4358
step 16000 | loss: 2.4180
step 17000 | loss: 2.4991
step 18000 | loss: 2.5522
step 19000 | loss: 2.5632
Final loss: 2.4484


In [14]:
## evaluate the model.

import math

@torch.no_grad()
def estimate_loss(eval_iters=200):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            logits, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

losses = estimate_loss()
print(f"train loss: {losses['train']:.4f} | perplexity: {math.exp(losses['train']):.2f}")
print(f"val loss  : {losses['val']:.4f}   | perplexity: {math.exp(losses['val']):.2f}")

train loss: 2.5106 | perplexity: 12.31
val loss  : 2.5600   | perplexity: 12.94


In [15]:
## sample next chars now.

# Quick greedy generation — shows the loop, hooks into Chapter 01c
def generate_greedy(model, start_char, max_new_tokens=50):
    model.eval()
    idx = torch.tensor([[stoi[start_char]]], dtype=torch.long).to(device)
    for _ in range(max_new_tokens):
        logits, _ = model(idx)
        next_idx = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        idx = torch.cat([idx, next_idx], dim=1)
    return decode(idx[0].tolist())

print("Greedy output:")
print(generate_greedy(model, 't'))
print()
print("→ Notice the loop. Chapter 01c fixes this with sampling strategies.")

Greedy output:
the the the the the the the the the the the the the

→ Notice the loop. Chapter 01c fixes this with sampling strategies.
